In [ ]:
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, recall_score,precision_score, roc_auc_score
from sklearn.decomposition import PCA
from collections import Counter
from collections import defaultdict
import time

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
def enhance_attack_features(df, target_attacks):
   
    df_enhanced = df.copy()

    df_enhanced['Flow_Duration'] = df_enhanced['Flow Duration'] / 1_000_000

    df_enhanced['Flow_Duration'] = df_enhanced['Flow_Duration'].replace(0, 1e-6)
    
    df_enhanced['Fwd_Bwd_Packet_Ratio'] = df_enhanced['Total Fwd Packet'] / (df_enhanced['Total Bwd packets'] + 1)
    df_enhanced['Fwd_Bwd_Bytes_Ratio'] = df_enhanced['Total Length of Fwd Packet'] / (df_enhanced['Total Length of Bwd Packet'] + 1)
  
    df_enhanced['Avg_Fwd_Packet_Size'] = df_enhanced['Total Length of Fwd Packet'] / (df_enhanced['Total Fwd Packet'] + 1)
    df_enhanced['Packet_Size_Variance_Ratio'] = df_enhanced['Packet Length Variance'] / (df_enhanced['Flow Bytes/s'] + 1)
    
   
    df_enhanced['ICMP_Flood_Score'] = (
        df_enhanced['Proto_17'] * 
        df_enhanced['Fwd Packets/s'] * 
        (df_enhanced['Total Fwd Packet'] > df_enhanced['Total Fwd Packet'].quantile(0.8)).astype(int)
    )
    

    df_enhanced['TCP_Flood_Score'] = (
        df_enhanced['Proto_6'] * 
        df_enhanced['Fwd Packets/s'] * 
        (df_enhanced['SYN Flag Count'] + df_enhanced['RST Flag Count'] + df_enhanced['FIN Flag Count'])
    )
    

    http_port_mask = (
        (df_enhanced['Dst Port'] == 80) | 
        (df_enhanced['Dst Port'] == 443) | 
        (df_enhanced['Dst Port'] == 8080) | 
        (df_enhanced['Dst Port'] == 8443)
    )
    df_enhanced['HTTP_Flood_Score'] = (
        http_port_mask.astype(int) * 
        df_enhanced['Fwd Packets/s'] * 
        (df_enhanced['PSH Flag Count'] + df_enhanced['Fwd PSH Flags'] + df_enhanced['Bwd PSH Flags'])
    )
    
    
    df_enhanced['Port_Scan_Score'] = (
        df_enhanced['Dst Port_freq'] *  
        df_enhanced['Src IP_freq'] *   
        (df_enhanced['Total Fwd Packet'] < df_enhanced['Total Fwd Packet'].quantile(0.5)).astype(int)  
    )
    
 
    df_enhanced['Network_Scan_Score'] = (
        df_enhanced['Dst IP_freq'] *    
        df_enhanced['Src IP_freq'] *   
        (df_enhanced['SYN Flag Count'] > 0).astype(int) *  
        (df_enhanced['Total Length of Bwd Packet'] == 0).astype(int)  
    )
    
    
    df_enhanced['IAT_Consistency'] = np.where(
        df_enhanced['Fwd IAT Std'] > 0,
        df_enhanced['Fwd IAT Mean'] / df_enhanced['Fwd IAT Std'],
        0
    )
    
    
    
    df_enhanced['DDoS_Combined_Score'] = (
        df_enhanced['Fwd Packets/s'] * 0.3 +
        df_enhanced['ICMP_Flood_Score'] * 0.2 +
        df_enhanced['TCP_Flood_Score'] * 0.2 +
        df_enhanced['HTTP_Flood_Score'] * 0.2 +
        df_enhanced['IAT_Consistency'] * 0.1
    )
    

    df_enhanced['Scan_Combined_Score'] = (
        df_enhanced['Port_Scan_Score'] * 0.4 +
        df_enhanced['Network_Scan_Score'] * 0.4 +
        (df_enhanced['Dst Port_freq'] * df_enhanced['Dst IP_freq']) * 0.2
    )
  
    df_enhanced['Flag_Usage_Score'] = (
        df_enhanced['SYN Flag Count'] + 
        df_enhanced['RST Flag Count'] + 
        df_enhanced['FIN Flag Count'] + 
        df_enhanced['PSH Flag Count'] + 
        df_enhanced['URG Flag Count']
    ) / (df_enhanced['Total Fwd Packet'] + 1)
    

    df_enhanced['Subflow_Anomaly'] = np.abs(
        df_enhanced['Subflow Fwd Bytes'] - df_enhanced['Subflow Bwd Bytes']
    ) / (df_enhanced['Flow Bytes/s'] + 1)
    
    
    attack_indicators = {}
    

    attack_indicators['DDoS_HTTP_Indicator'] = (
        df_enhanced['HTTP_Flood_Score'] * 
        (df_enhanced['Proto_6'] == 1).astype(int) * 
        (df_enhanced['Fwd Packets/s'] > df_enhanced['Fwd Packets/s'].quantile(0.9)).astype(int)
    )

    attack_indicators['DDoS_ICMP_Indicator'] = (
        df_enhanced['ICMP_Flood_Score'] *
        (df_enhanced['Bwd Packet Length Min'] == 0).astype(int) *  
        (df_enhanced['Fwd Packets/s'] > df_enhanced['Fwd Packets/s'].quantile(0.8)).astype(int)
    )
    
  
    attack_indicators['DDoS_ICMP_Frag_Indicator'] = (
        df_enhanced['Proto_17'] *  
        (df_enhanced['Packet Length Variance'] > df_enhanced['Packet Length Variance'].quantile(0.7)).astype(int) *
        df_enhanced['Fwd Packets/s']
    )
    

    attack_indicators['VulnScan_Indicator'] = (
        df_enhanced['Scan_Combined_Score'] *
        (df_enhanced['Active Min'] > 0).astype(int) *  
        (df_enhanced['Total Length of Bwd Packet'] < df_enhanced['Total Length of Fwd Packet']).astype(int)
    )
    

    for indicator_name, indicator_values in attack_indicators.items():
        df_enhanced[indicator_name] = indicator_values
    
    print("Enhanced features created:")
    print(f"- Basic rate features: Fwd Packets/s, Byte_Rate, etc.")
    print(f"- Protocol-specific scores: ICMP_Flood_Score, TCP_Flood_Score, HTTP_Flood_Score")
    print(f"- Scan detection scores: Port_Scan_Score, Network_Scan_Score")
    print(f"- Combined attack indicators: DDoS_Combined_Score, Scan_Combined_Score")
    print(f"- Target-specific indicators: {list(attack_indicators.keys())}")
    
    return df_enhanced

In [ ]:
def create_anomaly_label(row):
    if row['Attack_Type'] == 'Benign&Bruteforce_benign':
        return 'Normal'
    else:
        return 'Attack'

In [ ]:
def preprocess():
    
    df = pd.read_csv('NewDatasets/SmallerDataset_w_addedAttacks/increased_smaller_dataset.csv')

    df["Weight"] = df["Total Fwd Packet"] * df["Total Bwd packets"]

    categorical_columns = ["Src IP", 'Dst IP', "Src Port", "Dst Port", "Protocol"]
    for col in categorical_columns:
        df[col] = df[col].astype('category')

    for col in ['Src IP', 'Dst IP', 'Src Port', 'Dst Port']:
        df[col + '_freq'] = df[col].map(df[col].value_counts())

    df = pd.get_dummies(df, columns=['Protocol'], prefix='Proto')

    df['Timestamp'] = pd.to_datetime(df['Timestamp'],  format='%d/%m/%Y %I:%M:%S %p')

    attacks_to_remove = [
    "spoofing_ARP Spoofing",
    "spoofing_DNS Spoofing",
    "sqlinjection",
    "XSS",
    "Benign&Bruteforce_BruteForce",
    "Uploading_Attack"
    ]

    df = df[~df["Attack_Type"].isin(attacks_to_remove)]

    target_attacks = ['DDoS_DDoS-HTTP Flood', 'DDoS_DDoS ICMP Flood', 
                 'DDoS_DDoS-ICMP_Fragmentation', 'VulnerabilityScan']
    df = enhance_attack_features(df, target_attacks)

    numeric_columns = df.select_dtypes(include=['number'])

    corr_matrix = numeric_columns.corr()

    threshold = 0.9

    mask = (corr_matrix.abs() >= threshold) & (corr_matrix != 1)

    filtered_corr = corr_matrix[mask]

    printed_pairs = set()

    for col1 in filtered_corr.columns:
        for col2 in filtered_corr.index:
            if not pd.isna(filtered_corr.loc[col2, col1]):
                pair = tuple(sorted([col1, col2]))
                if pair not in printed_pairs:
                    printed_pairs.add(pair)
                    print(f"Columns: {col1} and {col2} | Correlation: {filtered_corr.loc[col2, col1]:.3f}")

    columns_to_drop = ["Flow IAT Std",
                   "Bwd Segment Size Avg",
                   "Subflow Fwd Packets",
        'Flow Duration',
        'Subflow Bwd Packets',
        'Fwd Packet Length Max',
        'Fwd Packet Length Min',
        'Flow Packets/s',
        'Flow IAT Min',
        'Flow IAT Max',
        'Bwd IAT Max',
        'Bwd IAT Min',
        'Fwd Header Length',
        'ACK Flag Count',
        'Packet Length Std',
        "Fwd IAT Max",
        "Idle Max",
        "Idle Min",
        "Fwd Packet Length Mean",
        "Bwd Packet Length Max",
        'Average Packet Size',
        'Fwd Segment Size Avg',
        'Fwd IAT Max',
        'Bwd Header Length',
        'Packet Length Mean',
        'CWR Flag Count',
        'Average Packet Size',
        "Flow IAT Mean",
        "Active Max",
        "Bwd Bytes/Bulk Avg",
        'Fwd IAT Mean',
        'Active Mean',
        'Active Std',
        "Fwd Act Data Pkts",
        "Flow_Duration",
        "ICMP_Flood_Score",
        "HTTP_Flood_Score",
        "Port_Scan_Score",
        "IAT_Consistency" ,
        "DDoS_Combined_Score"
    ]

    df = df.drop(columns=columns_to_drop)
    df = df.drop(columns="Label")

    df['Anomaly_Label'] = df.apply(create_anomaly_label, axis=1)

    null_columns = df.isnull().sum()
    null_columns = null_columns[null_columns > 0]
    inf_columns = df.columns[(df == np.inf).any() | (df == -np.inf).any()]
    df = df.dropna()
    df = df[~df.isin([np.inf, -np.inf]).any(axis=1)]
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    

    return df

In [ ]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, activation=nn.ReLU(), dropout_rate=0.0,
                 sequence_length=1, num_layers=1, bidirectional=False):
        super(LSTMAutoencoder, self).__init__()

        self.sequence_length = sequence_length
        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.bottleneck_dim = hidden_dims[-1]
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.directions = 2 if bidirectional else 1
        self.activation = activation

        self.encoder_layers = nn.ModuleList()
        prev_dim = input_dim
        for h_dim in hidden_dims:
            self.encoder_layers.append(nn.LSTM(
                input_size=prev_dim,
                hidden_size=h_dim,
                num_layers=1,
                batch_first=True,
                bidirectional=bidirectional
            ))
            prev_dim = h_dim * self.directions


        self.decoder_layers = nn.ModuleList()
        reversed_dims = list(reversed(hidden_dims))
        prev_dim = reversed_dims[0] * self.directions  

        for i in range(len(reversed_dims) - 1):
            next_dim = reversed_dims[i+1]
            self.decoder_layers.append(nn.LSTM(
                input_size=prev_dim,
                hidden_size=next_dim,
                num_layers=1,
                batch_first=True,
                bidirectional=bidirectional
            ))
            prev_dim = next_dim * self.directions


        self.output_layer = nn.Linear(hidden_dims[0] * self.directions, input_dim)

        self.dropout = nn.Dropout(dropout_rate) if dropout_rate > 0 else None

    def forward(self, x):

        original_shape = x.shape
        if len(x.shape) == 2:
            x = x.unsqueeze(1)  

        batch_size = x.size(0)
        seq_len = x.size(1)

        current_input = x
        for i, encoder in enumerate(self.encoder_layers):
            outputs, (hidden, cell) = encoder(current_input)
            current_input = outputs

            if self.activation is not None:
                current_input = self.activation(current_input)

            if self.dropout is not None and i < len(self.encoder_layers) - 1:
                current_input = self.dropout(current_input)

        encoded = outputs[:, -1, :]


        decoder_input = encoded.unsqueeze(1).repeat(1, seq_len, 1)

        current_input = decoder_input
        for i, decoder in enumerate(self.decoder_layers):
            outputs, _ = decoder(current_input)
            current_input = outputs

            if self.activation is not None and i < len(self.decoder_layers) - 1:
                current_input = self.activation(current_input)

            if self.dropout is not None and i < len(self.decoder_layers) - 1:
                current_input = self.dropout(current_input)

        reconstructed = self.output_layer(current_input)

        if len(original_shape) == 2:
            reconstructed = reconstructed.squeeze(1)

        return reconstructed, encoded

In [ ]:
def prepare_dataset(df):
    X = df.select_dtypes(include=['number', 'bool'])
    X_datetime = df.select_dtypes(include=['datetime64'])

    attack_types = df['Attack_Type'] 
    y = df['Anomaly_Label']  
    y_binary = (y == "Attack").astype(int)  

  
    df_temp = df.copy()
    df_temp['stratify_col'] = df_temp['Attack_Type'] 


    df_temp.loc[df_temp['Anomaly_Label'] == 'Normal', 'stratify_col'] = 'Normal'

    print("Original Attack Type Distribution:")
    print(df_temp['stratify_col'].value_counts())
    print(f"\nOriginal Attack/Normal Distribution:")
    print(df_temp['Anomaly_Label'].value_counts())

    np.random.seed(42)
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)


    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    for col in X_datetime.columns:
        X_datetime[col] = pd.to_datetime(X_datetime[col]).astype('int64') // 10**9


    scaler_numerical = StandardScaler()
    X_scaled_num = scaler_numerical.fit_transform(X)

    scaler_datetime = StandardScaler()
    X_scaled_datetime = scaler_datetime.fit_transform(X_datetime)

    X_scaled = np.concatenate([X_scaled_num, X_scaled_datetime], axis=1)
    print("Had ", X_scaled.shape[1], "features")


    pca = PCA(n_components=0.95) 
    X_scaled = pca.fit_transform(X_scaled)
    print("Components after reduction: ", pca.n_components_) 


    try:
        X_train_full, X_test, y_train_full, y_test, stratify_train, stratify_test = train_test_split(
            X_scaled, y_binary, df_temp['stratify_col'],
            test_size=0.3, random_state=42, stratify=df_temp['stratify_col']
        )

        original_indices = np.arange(len(X_scaled))
        train_indices, test_indices = train_test_split(
            original_indices, test_size=0.3, random_state=42, stratify=df_temp['stratify_col']
        )

        print("\n=== TRAIN SET DISTRIBUTION ===")
        train_stratify_counts = pd.Series(stratify_train).value_counts()
        print("Attack Type Distribution in Training Set:")
        print(train_stratify_counts)
        print(f"Normal vs Attack in Training Set:")
        print(pd.Series(y_train_full).value_counts())

        print("\n=== TEST SET DISTRIBUTION ===")
        test_stratify_counts = pd.Series(stratify_test).value_counts()
        print("Attack Type Distribution in Test Set:")
        print(test_stratify_counts)
        print(f"Normal vs Attack in Test Set:")
        print(pd.Series(y_test).value_counts())

    except ValueError as e:
        print(f"Stratification failed: {e}")
        print("Some attack types might have too few samples. Using alternative approach")

        attack_type_indices = defaultdict(list)
        for idx, attack_type in enumerate(df_temp['stratify_col']):
            attack_type_indices[attack_type].append(idx)

        train_indices = []
        test_indices = []

        for attack_type, indices in attack_type_indices.items():
            if len(indices) == 1:
        
                train_indices.extend(indices)
            else:

                n_test = max(1, int(len(indices) * 0.3))
                np.random.shuffle(indices)
                test_indices.extend(indices[:n_test])
                train_indices.extend(indices[n_test:])

        X_train_full = X_scaled[train_indices]
        X_test = X_scaled[test_indices]
        y_train_full = y_binary.iloc[train_indices].values
        y_test = y_binary.iloc[test_indices].values
        stratify_train = df_temp['stratify_col'].iloc[train_indices].values
        stratify_test = df_temp['stratify_col'].iloc[test_indices].values

        print("\n=== TRAIN SET DISTRIBUTION (Manual Split) ===")
        print("Attack Type Distribution in Training Set:")
        print(pd.Series(stratify_train).value_counts())
        print(f"Normal vs Attack in Training Set:")
        print(pd.Series(y_train_full).value_counts())

        print("\n=== TEST SET DISTRIBUTION (Manual Split) ===")
        print("Attack Type Distribution in Test Set:")
        print(pd.Series(stratify_test).value_counts())
        print(f"Normal vs Attack in Test Set:")
        print(pd.Series(y_test).value_counts())


    X_train = X_train_full[y_train_full == 0]


    attack_train_mask = (y_train_full == 1)
    X_test = np.concatenate([X_test, X_train_full[attack_train_mask]], axis=0)
    y_test = np.concatenate([y_test, y_train_full[attack_train_mask]], axis=0)

   
    original_test_indices = test_indices.copy()


    attack_train_indices = np.where(y_train_full == 1)[0]
    attack_train_original_indices = np.array(train_indices)[attack_train_indices]

    test_indices = np.concatenate([original_test_indices, attack_train_original_indices])

    print(f"Updated test set size: {len(y_test)}")
    print(f"Test indices size: {len(test_indices)}")
    assert len(test_indices) == len(y_test), "Array sizes don't match!"

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(
        y_test.values if hasattr(y_test, 'values') else y_test, 
        dtype=torch.bool
    )

    train_loader = DataLoader(TensorDataset(X_train_tensor), batch_size=64, shuffle=True)

    return train_loader, X_test_tensor, y_test_tensor, test_indices, y_test, X_train


In [ ]:
def initialize_lstm_autoencoder(
    input_dim: int,
    hidden_dims: list,
    sequence_length: int = 1,
    bidirectional: bool = False,
    dropout_rate: float = 0.0,
    seed: int = 42
):
   
  
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = LSTMAutoencoder(
        input_dim=input_dim,
        hidden_dims=hidden_dims,
        sequence_length=sequence_length,
        num_layers=1,
        bidirectional=bidirectional,
        dropout_rate=dropout_rate
    ).to(device)

    return model, device


In [ ]:
def train_and_evaluate(model, train_loader, X_test_tensor, y_test, device, n_epochs=50):

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
  
    train_losses = []
    test_losses = []

    start_train = time.time()
    model.train()
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        for batch in train_loader:
            inputs = batch[0].to(device)

            if len(inputs.shape) == 2:
                inputs = inputs.unsqueeze(1)  

            outputs, _ = model(inputs)  
            loss = criterion(outputs, inputs)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_train_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        end_train = time.time()
        train_time = end_train - start_train

        start_test = time.time()
        model.eval()
        with torch.no_grad():
            X_test_for_loss = X_test_tensor.to(device)
            if len(X_test_for_loss.shape) == 2:
                X_test_for_loss = X_test_for_loss.unsqueeze(1)
            outputs, _ = model(X_test_for_loss)
            test_loss = criterion(outputs, X_test_for_loss).item()
            test_losses.append(test_loss)

        
        scheduler.step(avg_train_loss)
        print(f"Epoch {epoch+1}/{n_epochs}, Train Loss: {avg_train_loss:.4f}, Test Loss: {test_loss:.4f}")
    end_test = time.time()
    test_time = end_test - start_test
    
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, n_epochs + 1), train_losses, label='Training Loss', marker='o')
    plt.plot(range(1, n_epochs + 1), test_losses, label='Test Loss', marker='o')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Training and Test Loss Over Time')
    plt.legend()
    plt.grid(True)
    plt.yscale('log')
    plt.tight_layout()
    plt.show()


    y_test_np = np.array(y_test)
    model.eval()
    with torch.no_grad():
        X_test = X_test_tensor.to(device)
        if len(X_test.shape) == 2:
            X_test = X_test.unsqueeze(1)
        reconstructions, _ = model(X_test)
        if len(X_test.shape) == 3:
            reconstruction_error = torch.mean((X_test - reconstructions) ** 2, dim=(1, 2)).cpu().numpy()
        else:
            reconstruction_error = torch.mean((X_test - reconstructions) ** 2, dim=1).cpu().numpy()

   
    best_f1 = 0
    best_threshold = 0
    for percentile in range(0, 100):
        threshold = np.percentile(reconstruction_error, percentile)
        y_pred = (reconstruction_error > threshold).astype(int)
        f1 = f1_score(y_test_np, y_pred)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    print(f"Optimal threshold: {best_threshold:.6f}, F1 score: {best_f1:.4f}")


    y_pred = (reconstruction_error > best_threshold).astype(int)
    print("Threshold used:", best_threshold)
    print("\nClassification Report:")
    print(classification_report(y_test_np, y_pred, target_names=["Normal", "Anomaly"]))
    print("Accuracy:", accuracy_score(y_test_np, y_pred))
    print("F1 Score:", f1_score(y_test_np, y_pred))
    print("Precision:", precision_score(y_test_np, y_pred))
    print("Recall:", recall_score(y_test_np, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test_np, reconstruction_error))
    print("Stage 1 train time:", train_time)
    print("Stage 1 test time:", test_time)

    cm = confusion_matrix(y_test_np, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Normal", "Anomaly"],
                yticklabels=["Normal", "Anomaly"])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Test Data Confusion Matrix")
    plt.show()

    return train_losses, test_losses, reconstruction_error, y_pred, best_threshold, best_f1


In [ ]:
def misclassification_analysis(y_test, y_pred, test_indices, df):
    
    print(f"Updated test set size: {len(y_test)}")
    print(f"Prediction array size: {len(y_pred)}")
    print(f"Test indices size: {len(test_indices)}")

    
    assert len(y_test) == len(y_pred) == len(test_indices), "Array sizes don't match!"

    false_negatives = (y_test == 1) & (y_pred == 0)  
    false_positives = (y_test == 0) & (y_pred == 1)  

   
    fn_indices = test_indices[false_negatives]
    fp_indices = test_indices[false_positives]

    print("\n----- MISCLASSIFICATION ANALYSIS -----")
    print(f"Number of false negatives (missed attacks): {np.sum(false_negatives)}")
    print(f"Number of false positives (false alarms): {np.sum(false_positives)}")

   
    print("\n----- TEST SET ANALYSIS -----")
    test_df = df.iloc[test_indices].copy()
    test_df['predicted'] = y_pred
    test_df['actual'] = y_test

 
    attack_types_in_test = test_df[test_df['Anomaly_Label'] == 'Attack']['Attack_Type'].unique()

    print("\nDetection rate by attack type (sorted from highest to lowest):")

    detection_rates_test_analysis = []
    for attack_type in attack_types_in_test:
        attack_subset = test_df[(test_df['Attack_Type'] == attack_type) & 
                                (test_df['Anomaly_Label'] == 'Attack')]

        if len(attack_subset) > 0:
            total_attacks = len(attack_subset)
            detected_attacks = np.sum(attack_subset['predicted'] == 1)
            detection_rate = (detected_attacks / total_attacks) * 100
            detection_rates_test_analysis.append((attack_type, detection_rate, detected_attacks, total_attacks))

  
    detection_rates_test_analysis.sort(key=lambda x: x[1], reverse=True)


    for attack_type, detection_rate, detected_attacks, total_attacks in detection_rates_test_analysis:
        print(f"{attack_type}: {detection_rate:.2f}% detected ({detected_attacks}/{total_attacks})")

    return fn_indices, fp_indices, detection_rates_test_analysis


In [ ]:
def train_second_stage_lstm(model_stage1, X_test_tensor, y_test, test_indices, df, device, best_threshold_stage1, best_f1_stage1,
                            target_attacks=['DDoS_DDoS-HTTP Flood', 'DDoS_DDoS ICMP Flood', 
                                          'DDoS_DDoS-ICMP_Fragmentation', 'VulnerabilityScan', 'Benign&Bruteforce_BruteForce'],
                            n_epochs=50):
   
    
    print("="*60)
    print("STARTING SECOND STAGE TRAINING")
    print("="*60)
    
    model_stage1.eval()
    with torch.no_grad():
        X_test = X_test_tensor.to(device)
        if len(X_test.shape) == 2:
            X_test = X_test.unsqueeze(1)
        reconstructions_stage1, _ = model_stage1(X_test)
        if len(X_test.shape) == 3:
            reconstruction_error_stage1 = torch.mean((X_test - reconstructions_stage1) ** 2, dim=(1, 2)).cpu().numpy()
        else:
            reconstruction_error_stage1 = torch.mean((X_test - reconstructions_stage1) ** 2, dim=1).cpu().numpy()
    
    y_pred_stage1 = (reconstruction_error_stage1 > best_threshold_stage1).astype(int)
    
    print(f"Stage 1 - Threshold: {best_threshold_stage1:.6f}, F1: {best_f1_stage1:.4f}")

    stage1_benign_mask = (y_pred_stage1 == 0)
    
    print(f"Stage 1 labeled {np.sum(stage1_benign_mask)} samples as benign out of {len(y_test)} total")
    
    stage1_benign_indices = test_indices[stage1_benign_mask]
    stage1_benign_df = df.iloc[stage1_benign_indices].copy()
    
    y_test_np = np.array(y_test)

    target_attack_mask = stage1_benign_df['Attack_Type'].isin(target_attacks)
    missed_target_attacks = stage1_benign_df[target_attack_mask]
    
    print(f"\\nTarget attacks missed by Stage 1:")
    for attack in target_attacks:
        count = np.sum(stage1_benign_df['Attack_Type'] == attack)
        if count > 0:
            print(f"  {attack}: {count} samples")
    
    print(f"\\nTotal target attack samples for Stage 2 training: {len(missed_target_attacks)}")
    
 
    actual_benign_in_stage1_benign = stage1_benign_df[stage1_benign_df['Anomaly_Label'] == 'Normal']
    
    stage1_benign_tensor_mask = np.zeros(len(X_test_tensor), dtype=bool)
    stage1_benign_tensor_mask[np.where(stage1_benign_mask)[0]] = True
    
    X_stage2_candidates = X_test_tensor[stage1_benign_tensor_mask]
    y_stage2_candidates = y_test_np[stage1_benign_mask]
    
    actual_normal_mask_in_stage2 = (y_stage2_candidates == 0)
    X_stage2_train = X_stage2_candidates[actual_normal_mask_in_stage2]
    
    print(f"Stage 2 training on {len(X_stage2_train)} normal samples")
    print(f"Stage 2 will test on {len(X_stage2_candidates)} stage-1-benign samples")
    
    if len(X_stage2_train) < 10:
        print("WARNING: Very few training samples for Stage 2. Consider adjusting thresholds.")
        return None, None, y_pred_stage1, {}
    

    input_dim_stage2 = X_stage2_train.shape[1] if len(X_stage2_train.shape) == 2 else X_stage2_train.shape[2]
    hidden_dims_stage2 = [16, 8]  
    
    model_stage2, _ = initialize_lstm_autoencoder(
        input_dim=input_dim_stage2,
        hidden_dims=hidden_dims_stage2,
        sequence_length=1, 
        bidirectional=False,
        dropout_rate=0.05,
        seed=42
    )

    X_stage2_train_tensor = torch.tensor(X_stage2_train, dtype=torch.float32)
    stage2_train_loader = DataLoader(TensorDataset(X_stage2_train_tensor), batch_size=32, shuffle=True)
    
    print(f"\\nTraining Stage 2 model for {n_epochs} epochs...")
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model_stage2.parameters(), lr=5e-4)  
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.7)
    
    stage2_train_losses = []
    start_train = time.time()
    model_stage2.train()
    for epoch in range(n_epochs):
        epoch_loss = 0
        for batch in stage2_train_loader:
            inputs = batch[0].to(device)
            if len(inputs.shape) == 2:
                inputs = inputs.unsqueeze(1)
            
            outputs, _ = model_stage2(inputs)
            loss = criterion(outputs, inputs)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(stage2_train_loader)
        stage2_train_losses.append(avg_loss)
        scheduler.step(avg_loss)
        
        if (epoch + 1) % 3 == 0:
            print(f"Stage 2 Epoch {epoch+1}/{n_epochs}, Loss: {avg_loss:.6f}")
    end_train = time.time()
    train_time = end_train - start_train

    start_test = time.time()

    model_stage2.eval()
    with torch.no_grad():
        X_stage2_test = X_stage2_candidates.to(device)
        if len(X_stage2_test.shape) == 2:
            X_stage2_test = X_stage2_test.unsqueeze(1)
        reconstructions_stage2, _ = model_stage2(X_stage2_test)
        if len(X_stage2_test.shape) == 3:
            reconstruction_error_stage2 = torch.mean((X_stage2_test - reconstructions_stage2) ** 2, dim=(1, 2)).cpu().numpy()
        else:
            reconstruction_error_stage2 = torch.mean((X_stage2_test - reconstructions_stage2) ** 2, dim=1).cpu().numpy()
    end_test = time.time()
    test_time = end_test - start_test
    
    best_f1_stage2 = 0
    best_threshold_stage2 = 0

    y_stage2_eval = np.zeros(len(y_stage2_candidates))
    stage2_eval_df = df.iloc[stage1_benign_indices].copy()
    target_attack_mask_eval = stage2_eval_df['Attack_Type'].isin(target_attacks)
    y_stage2_eval[target_attack_mask_eval] = 1
    
    print(f"\\nStage 2 evaluation: {np.sum(y_stage2_eval)} target attacks out of {len(y_stage2_eval)} samples")
    
    if np.sum(y_stage2_eval) > 0:  
        for percentile in range(50, 99): 
            threshold = np.percentile(reconstruction_error_stage2, percentile)
            y_pred_stage2_temp = (reconstruction_error_stage2 > threshold).astype(int)
            
            if np.sum(y_pred_stage2_temp) > 0:  
                f1 = f1_score(y_stage2_eval, y_pred_stage2_temp, zero_division=0)
                if f1 > best_f1_stage2:
                    best_f1_stage2 = f1
                    best_threshold_stage2 = threshold
    
    print(f"Stage 2 - Threshold: {best_threshold_stage2:.6f}, F1: {best_f1_stage2:.4f}")
  
    y_pred_stage2_on_stage1_benign = (reconstruction_error_stage2 > best_threshold_stage2).astype(int)
    
    print(f"\n" + "="*60)
    print("DETAILED STAGE 2 PERFORMANCE METRICS")
    print("="*60)
    
    stage2_accuracy = accuracy_score(y_stage2_eval, y_pred_stage2_on_stage1_benign)
    stage2_precision = precision_score(y_stage2_eval, y_pred_stage2_on_stage1_benign, zero_division=0)
    stage2_recall = recall_score(y_stage2_eval, y_pred_stage2_on_stage1_benign, zero_division=0)
    stage2_f1 = f1_score(y_stage2_eval, y_pred_stage2_on_stage1_benign, zero_division=0)
    
    print("\nStage 2 Standalone Metrics:")
    print(f"Accuracy:  {stage2_accuracy:.4f}")
    print(f"Precision: {stage2_precision:.4f}")
    print(f"Recall:    {stage2_recall:.4f}")
    print(f"F1-Score:  {stage2_f1:.4f}")
    
    print("\nStage 2 Confusion Matrix:")
    cm_stage2 = confusion_matrix(y_stage2_eval, y_pred_stage2_on_stage1_benign)
    
    tn_s2, fp_s2, fn_s2, tp_s2 = cm_stage2.ravel() if cm_stage2.size == 4 else (0, 0, 0, 0)
    total_stage2 = len(y_stage2_eval)
    
    print(f"                 Predicted")
    print(f"                 Benign  Attack")
    print(f"Actual Benign    {tn_s2:4d}    {fp_s2:4d}     (TN: {tn_s2/total_stage2*100:.1f}%, FP: {fp_s2/total_stage2*100:.1f}%)")
    print(f"Actual Attack    {fn_s2:4d}    {tp_s2:4d}     (FN: {fn_s2/total_stage2*100:.1f}%, TP: {tp_s2/total_stage2*100:.1f}%)")
    print(f"Total: {total_stage2} samples")
    
    print(f"\nStage 2 Detection Analysis:")
    print(f"True Positives (Correctly identified target attacks): {tp_s2} ({tp_s2/total_stage2*100:.1f}%)")
    print(f"False Positives (Normal classified as attack): {fp_s2} ({fp_s2/total_stage2*100:.1f}%)")
    print(f"True Negatives (Correctly identified normal): {tn_s2} ({tn_s2/total_stage2*100:.1f}%)")
    print(f"False Negatives (Missed target attacks): {fn_s2} ({fn_s2/total_stage2*100:.1f}%)")

    print(f"\nStage 2 Target Attack Detection Breakdown:")
    for attack_type in target_attacks:
        attack_mask_s2 = stage2_eval_df['Attack_Type'] == attack_type
        if np.sum(attack_mask_s2) > 0:
            attack_indices = np.where(attack_mask_s2)[0]
            total_attack = len(attack_indices)
            detected_attack = np.sum(y_pred_stage2_on_stage1_benign[attack_indices] == 1)
            detection_rate = (detected_attack / total_attack) * 100
            
            print(f"  {attack_type}:")
            print(f"    Total: {total_attack}, Detected: {detected_attack}, Rate: {detection_rate:.1f}%")
    
    combined_predictions = y_pred_stage1.copy()
    
    stage1_benign_indices_in_combined = np.where(stage1_benign_mask)[0]
    combined_predictions[stage1_benign_indices_in_combined] = y_pred_stage2_on_stage1_benign
    
    print(f"\n" + "="*60)
    print("COMBINED TWO-STAGE RESULTS")
    print("="*60)
    
    print("\nCombined Classification Report:")
    print(classification_report(y_test_np, combined_predictions, target_names=["Normal", "Attack"]))
    
    combined_metrics = {
        'accuracy': accuracy_score(y_test_np, combined_predictions),
        'f1': f1_score(y_test_np, combined_predictions),
        'precision': precision_score(y_test_np, combined_predictions),
        'recall': recall_score(y_test_np, combined_predictions),
        'roc_auc': roc_auc_score(y_test_np, combined_predictions)
    }
    
    for metric, value in combined_metrics.items():
        print(f"Combined {metric.title()}: {value:.4f}")
    
    print("\nCombined Confusion Matrix:")
    cm_combined = confusion_matrix(y_test_np, combined_predictions)
    tn_comb, fp_comb, fn_comb, tp_comb = cm_combined.ravel()
    total_combined = len(y_test_np)
    
    print(f"                 Predicted")
    print(f"                 Benign  Attack")
    print(f"Actual Benign    {tn_comb:4d}    {fp_comb:4d}     (TN: {tn_comb/total_combined*100:.1f}%, FP: {fp_comb/total_combined*100:.1f}%)")
    print(f"Actual Attack    {fn_comb:4d}    {tp_comb:4d}     (FN: {fn_comb/total_combined*100:.1f}%, TP: {tp_comb/total_combined*100:.1f}%)")
    print(f"Total: {total_combined} samples")
    
    print(f"\n" + "-"*50)
    print("IMPROVEMENT ANALYSIS")
    print("-"*50)
    
    stage1_metrics = {
        'accuracy': accuracy_score(y_test_np, y_pred_stage1),
        'f1': f1_score(y_test_np, y_pred_stage1),
        'precision': precision_score(y_test_np, y_pred_stage1),
        'recall': recall_score(y_test_np, y_pred_stage1)
    }
    
    print("\nStage 1 Confusion Matrix (for comparison):")
    cm_stage1 = confusion_matrix(y_test_np, y_pred_stage1)
    tn_s1, fp_s1, fn_s1, tp_s1 = cm_stage1.ravel()
    
    print(f"                 Predicted")
    print(f"                 Benign  Attack")
    print(f"Actual Benign    {tn_s1:4d}    {fp_s1:4d}     (TN: {tn_s1/total_combined*100:.1f}%, FP: {fp_s1/total_combined*100:.1f}%)")
    print(f"Actual Attack    {fn_s1:4d}    {tp_s1:4d}     (FN: {fn_s1/total_combined*100:.1f}%, TP: {tp_s1/total_combined*100:.1f}%)")
    
    print("\nMetric Improvements:")
    for metric in ['accuracy', 'f1', 'precision', 'recall']:
        improvement = combined_metrics[metric] - stage1_metrics[metric]
        print(f"{metric.title()}: {stage1_metrics[metric]:.4f} → {combined_metrics[metric]:.4f} ({improvement:+.4f})")
    
    print(f"\nDetection Rate Improvements by Target Attack Type:")
    test_df_combined = df.iloc[test_indices].copy()
    test_df_combined['stage1_pred'] = y_pred_stage1
    test_df_combined['combined_pred'] = combined_predictions
    test_df_combined['actual'] = y_test_np
    
    for attack_type in target_attacks:
        attack_subset = test_df_combined[
            (test_df_combined['Attack_Type'] == attack_type) & 
            (test_df_combined['Anomaly_Label'] == 'Attack')
        ]
        
        if len(attack_subset) > 0:
            total = len(attack_subset)
            stage1_detected = np.sum(attack_subset['stage1_pred'] == 1)
            combined_detected = np.sum(attack_subset['combined_pred'] == 1)
            
            stage1_rate = (stage1_detected / total) * 100
            combined_rate = (combined_detected / total) * 100
            improvement = combined_rate - stage1_rate
            
            print(f"{attack_type}:")
            print(f"  Stage 1: {stage1_rate:.2f}% ({stage1_detected}/{total})")
            print(f"  Combined: {combined_rate:.2f}% ({combined_detected}/{total})")
            print(f"  Improvement: {improvement:+.2f} percentage points")

    stage2_metrics = {
        'threshold': best_threshold_stage2,
        'f1_score': best_f1_stage2,
        'accuracy': stage2_accuracy,
        'precision': stage2_precision,
        'recall': stage2_recall,
        'train_losses': stage2_train_losses,
        'samples_processed': len(X_stage2_candidates),
        'training_samples': len(X_stage2_train),
        'confusion_matrix': {
            'tn': int(tn_s2), 'fp': int(fp_s2), 'fn': int(fn_s2), 'tp': int(tp_s2)
        },
        'confusion_matrix_percentages': {
            'tn_pct': tn_s2/total_stage2*100,
            'fp_pct': fp_s2/total_stage2*100,
            'fn_pct': fn_s2/total_stage2*100,
            'tp_pct': tp_s2/total_stage2*100
        }
        
    }
    print("****************************\nStage 2 train time:", train_time)
    print("Stage 2 test time:", test_time)
    
    return model_stage2, best_threshold_stage2, combined_predictions, stage2_metrics


def run_two_stage_experiment(df):
    
    print("Starting Two-Stage LSTM Experiment...")
    
    train_loader, X_test_tensor, y_test_tensor, test_indices, y_test, X_train = prepare_dataset(df)
    
    input_dim = X_train.shape[1]
    hidden_dims = [24, 12]
    sequence_length = 5
    bidirectional = True
    dropout_rate = 0.1
    
    model_stage1, device = initialize_lstm_autoencoder(
        input_dim=input_dim,
        hidden_dims=hidden_dims,
        sequence_length=sequence_length,
        bidirectional=bidirectional,
        dropout_rate=dropout_rate,
        seed=42
    )
    
    print("Training Stage 1...")
    train_losses_s1, test_losses_s1, reconstruction_error_s1, y_pred_s1, threshold_s1, f1_s1 = train_and_evaluate(
        model_stage1, train_loader, X_test_tensor, y_test, device, n_epochs=10
    )
    
    print("Analyzing Stage 1 misclassifications...")
    fn_idx_s1, fp_idx_s1, detection_rates_s1 = misclassification_analysis(y_test, y_pred_s1, test_indices, df)
    
    print("\\nTraining Stage 2...")
    model_stage2, threshold_s2, combined_predictions, stage2_metrics = train_second_stage_lstm(
        model_stage1, X_test_tensor, y_test, test_indices, df, device, n_epochs=15
    )
    
    if combined_predictions is not None:
        print("\\nFinal combined analysis...")
        fn_idx_combined, fp_idx_combined, detection_rates_combined = misclassification_analysis(
            y_test, combined_predictions, test_indices, df
        )
        
        return {
            'stage1_model': model_stage1,
            'stage2_model': model_stage2,
            'stage1_threshold': threshold_s1,
            'stage2_threshold': threshold_s2,
            'stage1_predictions': y_pred_s1,
            'combined_predictions': combined_predictions,
            'stage1_metrics': {'f1': f1_s1, 'train_losses': train_losses_s1},
            'stage2_metrics': stage2_metrics,
            'detection_rates_stage1': detection_rates_s1,
            'detection_rates_combined': detection_rates_combined
        }
    
    return None

In [ ]:
df = preprocess()

In [ ]:
train_loader, X_test_tensor, y_test_tensor, test_indices, y_test, X_train = prepare_dataset(df)

input_dim = X_train.shape[1] 
hidden_dims = [24, 12]
sequence_length = 5
bidirectional = True
dropout_rate = 0.1

model, device = initialize_lstm_autoencoder(
    input_dim=input_dim,
    hidden_dims=hidden_dims,
    sequence_length=sequence_length,
    bidirectional=bidirectional,
    dropout_rate=dropout_rate,
    seed=42
)

batch_size = 32
x = torch.randn(batch_size, sequence_length, input_dim, device=device)
outputs, encoded = model(x)
print("Forward pass successful: outputs.shape =", outputs.shape)

train_losses, test_losses, reconstruction_error, y_pred, best_threshold, best_f1 = train_and_evaluate(
    model, train_loader, X_test_tensor, y_test, device, n_epochs=50
)


fn_idx, fp_idx, detection_rates = misclassification_analysis(y_test, y_pred, test_indices, df)

model_stage2, threshold_s2, combined_predictions, stage2_metrics = train_second_stage_lstm(
    model, X_test_tensor, y_test, test_indices, df, device, best_threshold, best_f1, n_epochs=50
)
